# Exploring GWAS summary statistics

**Purpose.** Most published GWAS release **summary statistics**, not genotypes: one row per
SNP with an effect size, a standard error and a p-value. Almost everything done afterwards —
meta-analysis, LD score regression, polygenic scores, Mendelian randomization — starts from
that file rather than from individual data. This exercise gets you used to reading one.

**What you will do**
 - read a published summary-statistics file and work out what each column means
 - filter on allele frequency and see how many variants remain
 - plot the results and locate the known signals
 - see what information is present, and what is irretrievably lost, compared with genotypes

**The data.** Published type 2 diabetes summary statistics from **Mahajan et al. (2018),
*Nature Genetics*** — a European meta-analysis of UK Biobank and HRC-imputed cohorts,
fine-mapping T2D loci to single-variant resolution.

**Summary statistics only — there are no individual genotypes in this exercise.** That is
the point: you never see a single person's data, which is precisely why summary statistics
can be shared when genotypes cannot.

# Exercise 2 - Exploring summary stats


We will use summary stats from 

Mahajan A, et al. (2018). Fine-mapping type 2 diabetes loci to single-variant 
resolution using high-density imputation and islet-specific epigenome maps. Nature Genetics http://dx.doi.org/10.1038/s41588-018-0241-6.

with summary stats available from 
https://diagram-consortium.org/downloads.html





First let's copy the summary stats to your folder:


In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data moves, this is the ONLY cell you need to change.
# No cell below this one uses a full path.
#############################################################

# where the summary statistics live (796 MB, so we read it in place
# rather than copying it into your home directory)
DATA=/course/data/current_data/novo23_gwas/sumstats
SUMSTATS=$DATA/Mahajan.NatGenet2018b.UKBB.HRC.T2D.European.txt

# the shared R plotting helpers
SCRIPTS=/course/data/current_data/scripts

# where you will do the exercise
WORK=$HOME/gwas_sumstats_human
mkdir -p $WORK
cd $WORK

cat > $WORK/env.sh <<EOF
export DATA=$DATA
export SCRIPTS=$SCRIPTS
export SUMSTATS=$SUMSTATS
export WORK=$WORK
cd $WORK
EOF

echo "working folder: $(pwd)"
echo
ls -lh $SUMSTATS

In [ ]:
# R cannot source env.sh, so read the paths out of it rather than repeating them
env <- readLines(path.expand("~/gwas_sumstats_human/env.sh"))
getvar <- function(k) sub(paste0('^export ', k, '='), '', grep(paste0('^export ', k, '='), env, value = TRUE)[1])
DATA    <- getvar("DATA")
SCRIPTS <- getvar("SCRIPTS")
WORK    <- getvar("WORK")
setwd(WORK)
getwd()


Now count the number of lines in the file:

In [ ]:
source ~/gwas_sumstats_human/env.sh

#count the number of lines
wc -l $SUMSTATS 

- How many variants are in the file? Compare that with the ~500,000 genotyped SNPs of the previous exercise — why does a published GWAS report so many more?

Let's first have a look at the first 10 lines of the file:

In [ ]:
source ~/gwas_sumstats_human/env.sh

head $SUMSTATS | column -t

- Each row is one SNP, with an effect size and its standard error. What could you *not* work out from this file that you could from the genotype data in exercise 1?

The different columns correspond to: 
- SNP ID
- Chromosome
- Position
- Effect Allele
- Non Effect Allele
- Effect allele frequency
- Beta (effect Size)
- Standard error of effect size
- Pvalue
- Number of individuals

How many variants are in your file?





To reduce the computational burden in this exercise, let's only look at the common variants with MAF>1% and remove some of the colums we don't need: 

In [ ]:
source ~/gwas_sumstats_human/env.sh

##cut columns 2,3,6,7,9 and remove site with low minor allele frequency
cut -f 2,3,6,7,9 $SUMSTATS | \
awk 'NR==1 || ($3<0.99 && $3>0.01)' > Mahajan.maf1.txt

head Mahajan.maf1.txt

echo "number of lines"
wc -l Mahajan.maf1.txt


- We kept only variants with a minor allele frequency above 1%. What happens to the effect estimate of a variant seen in only a handful of individuals?
- Which of the discarded columns would you have needed if you wanted to meta-analyse this study with another one?

Now make a manhattan plot in R:

In [ ]:

# read in the data 
d<-data.table::fread("Mahajan.maf1.txt")

# source R function for plotting
source(file.path(SCRIPTS, "newPlotPlink.R"))

# make manhattan plot
manPlot(d$Pvalue,d$Chr,cap=1e-30)

*If the plot does not show up, then rerun above cell*
 - How does the Manhattan plot look? 
 - How many association signals are there?

Try to change the cap option above in order to better see the number of peaks. e.g. cap=1e-30. 

And then try to make a QQ plot:

In [ ]:

source(file.path(SCRIPTS, "newPlotPlink.R"))

qqPlot(d$Pvalue,cap=1e-200) # <- try to change the cap

- How does the QQ plot look? Anything we should be worried about?

Try to change the cap option above in order to better see the shape of the QQ plot. e.g. cap=1e-30. 

# Highly polygenetic

As briefly mentioned in the lecture that are special issues one should consider if you have a very large sample size and if a trait is highly polygenic. In this case many of the SNPs in the genome will be in LD with a causal variant which will cause the QQ plot to look inflated. 

There is a nice method to evaluate this - called LD score regression - which we will be briefly mentioned. The output for this analysis 

**Total Liability scale h2: 0.0467 (0.0028)**

**Lambda GC: 1.3101**

**Mean Chi$^2$: 1.4606**

**Intercept: 1.0649 (0.0116)**

**Ratio: 0.1409 (0.0252)**

Indicates that most of the inflation observed in the qq plot is due to the trait being highly polygenetic. We will get back to these numbers later :-)

# Highest peak
Let's have a look at the highest peak. First let's have a look at the top SNP:

In [ ]:
w <- which.min(d$Pvalue)
d[w,]

The effect estimate (Beta) is the regression coefficient from the logistic regression, which is the logarithm of the odds ratio. Also, the measured effect allele is the common allele. We can convert to odds ratio using the exponential function:

In [ ]:
# the beta of the top SNP, taken from the row printed above
beta <- d$Beta[w]
cat("beta =", beta, "\n")

## convert to an odds ratio
OR <- exp(beta)
cat("OR =", OR, "\n")

## and for the minor allele, which is the other allele here
cat("OR for the minor allele =", exp(-beta), "\n")

- Try to estimate the odds ratio for the minor allele (change the sign of the effect size).

### Plot region

Let's visualize the region and see if we can identify the gene and causal variant:

In [ ]:
#select sites to plot, 0.5Mb on either side of SNP
pos <- 114758349
chr <- 10
win <- 5e5 # <- change this to zoom in

region <- subset(d,Chr==chr & Pos > pos-win &  Pos < pos+win)

#plot
locusZoomNoLD(region$Pvalue,chr=chr,pos=region$Pos,main="LocusZoom")

- Based on the plot what is the candidate gene?
- Can we be sure this is the causal gene?
- How many candidates are there for the causal allele (you can zoom in by changing the window size in the R code above and then rerunning the code)?

## overview of signals
Lets try to summaries different association signals. For simplicity lets assume that each peak represents only a single causal SNP. The below code will define a peak as being 1Mb and it will print the number of peaks

In [ ]:
## choose width of association peaks
win <- 1e6
g<-as.data.frame(subset(d,Pvalue<5e-8))


leadSNPs <- c()
for(i in 1:100){
    # stop when no significant variants are left, rather than
    # always looping 100 times
    if(nrow(g)==0) break
    #find top SNP
    w <- which.min(g$Pvalue)
    leadSNPs <- rbind(leadSNPs,g[w,])
    #remove region around top SNP
    g <- subset(g,Chr!=g[w,"Chr"] | Pos < g[w,"Pos"] -win |  Pos > g[w,"Pos"]+win)
}

cat("number of peaks")
nrow(leadSNPs)


- The loop takes the most significant SNP, removes everything within 1 Mb of it, and repeats. Why is the window needed at all — what makes the neighbouring SNPs significant too?
- Try changing `win`. Does the number of peaks change a lot, and what does that tell you about where to draw the line between one signal and two?

Print table of peaks on chr 3 

In [ ]:
subset(leadSNPs,Chr==3)

 one of the first variants known to be associated with type 2 diabetes is found on this chromosome in a gene located around position 12.3Mb. You can see the region with the following code

In [ ]:
#select sites to plot, 0.5Mb on either side of SNP

pos <- 12329783
chr <- 3
win <- 5e5
region <- subset(d,Chr==chr & Pos > pos-win &  Pos < pos+win)

#plot
locusZoomNoLD(region$Pvalue,chr=chr,pos=region$Pos,main="LocusZoom")

- Compare this region with the one you plotted before. Is the peak narrow or broad, and what does its width say about how precisely the causal variant is located?

We will futher explore this association signal using online resources.

However, fell free to explore other top signals and identify the gene by changing the above code (for example the top peak on chromosome 16). E.g. instead of chromosome 3 get the table of peaks for chromosome 16 and use its position to make a locuszoom plot.

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/gwas/quiz/gwas_sumstats.json")
